# Run the original GSM8K experiment on an A100

**Scope:** this launches the implemented GSM8K / Qwen2.5-7B-Instruct experiment. The repository also contains a revised MATH / Qwen3.5-9B proposal with a released lens; that revision is not implemented here. This first run uses the existing implementation to verify the full pipeline before a separate MATH migration.

This notebook launches the existing experiment from Jupyter, Colab, or another Linux cloud notebook. It uses its own Python 3.12.11 environment; the notebook kernel does not need to import the training stack.

**Before starting:** enable an NVIDIA GPU runtime and upload `jreflect-source.tar.gz` next to this notebook. In Colab, use **Runtime → Change runtime type → GPU**, then select the A100 if available. The hardware cell reports whether yours has 40 GB or 80 GB. Those are the planned memory sizes; full-model fitting has not yet been measured on either.

Run cells **one at a time**. First verify CUDA and ten problems. Then fit the lens, check its signal, train the three conditions, review dev results, and finally freeze/test. Defaults are three seeds and nine adapters. The expensive stages can consume substantial GPU time.

The package passed 39 local tests, including a tiny Qwen2 J++ fit and LoRA resume. This cloud notebook's full GPU sequence has not been executed locally. A tiny-model test checks mechanics, not scientific success.


## 1. Locations and hardware

Use a new run name for this notebook environment. Do not upload the Mac `.venv` or generation caches. `WORK_ROOT` must stay at the same absolute path when restoring a run because upstream Jacobian checkpoints contain paths.

For an attached persistent POSIX disk, set `WORK_ROOT` to that disk before continuing. On Colab, `/content` is temporary; use the backup section below.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import io
import json
import os
import shutil
import subprocess
import sys
import tarfile

WORK_ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
WORK_ROOT = WORK_ROOT.resolve()
PROJECT = WORK_ROOT / 'jreflect'
ARCHIVE = WORK_ROOT / 'jreflect-source.tar.gz'
RUN = PROJECT / 'artifacts' / 'notebook-run'
BACKUP_DIR = None  # Or Path('/your/persistent/volume/jreflect-backups')
print('Project:', PROJECT)
print('Run:', RUN)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)


## 2. Upload and extract the source

Colab prompts for the source archive if it is missing. In another notebook service, upload it through the file browser to `ARCHIVE`. Extraction uses only relative, regular files and directories. An existing project is reused rather than overwritten.


In [ ]:
def extract_regular_files(bundle, destination, prefix=None, members=None):
    destination = Path(destination).resolve()
    for member in (bundle.getmembers() if members is None else members):
        relative = PurePosixPath(member.name)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError(f'Unsafe archive path: {member.name}')
        if prefix is not None and relative.parts[:len(prefix)] != tuple(prefix):
            raise ValueError(f'Unexpected archive entry: {member.name}')
        target = destination.joinpath(*relative.parts)
        if not target.resolve().is_relative_to(destination):
            raise ValueError('Archive target escapes the destination')
        if member.isdir():
            target.mkdir(parents=True, exist_ok=True)
        elif member.isfile():
            target.parent.mkdir(parents=True, exist_ok=True)
            with bundle.extractfile(member) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)
        else:
            raise ValueError(f'Links and special files are not supported: {member.name}')

if not (PROJECT / 'pyproject.toml').exists():
    if not ARCHIVE.exists():
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(f'Upload the source bundle to {ARCHIVE}, then rerun this cell.') from exc
        uploaded = files.upload()
        if ARCHIVE.name not in uploaded:
            raise ValueError(f'Select {ARCHIVE.name}')
        ARCHIVE.write_bytes(uploaded[ARCHIVE.name])
    with tarfile.open(ARCHIVE, 'r:gz') as bundle:
        extract_regular_files(bundle, WORK_ROOT, prefix=('jreflect',))
else:
    print('Reusing the existing project; source files were not replaced.')
assert (PROJECT / 'src/jreflect/cli.py').is_file()


## 3. Install the isolated environment and pinned lens source

This installs the locked packages into `jreflect/.venv`, not into the notebook kernel. It pins Python 3.12.11 so a recreated session does not silently change Python versions. Keep package versions and source unchanged once a run starts.

The lockfile includes CUDA 13 dependencies on Linux. GPU model alone does not establish driver compatibility; the next cell checks an actual CUDA operation. If it fails, share that output and `nvidia-smi` before changing the environment or preparing a run.


In [ ]:
ENV = os.environ.copy()
ENV.setdefault('UV_CACHE_DIR', str(PROJECT / '.cache/uv'))
ENV.setdefault('HF_HOME', str(PROJECT / '.cache/huggingface'))
ENV.setdefault('MPLCONFIGDIR', str(PROJECT / '.cache/matplotlib'))
ENV.setdefault('XDG_CACHE_HOME', str(PROJECT / '.cache'))
ENV['PYTHONUNBUFFERED'] = '1'
ENV['TOKENIZERS_PARALLELISM'] = 'false'

def command(arguments):
    arguments = [str(argument) for argument in arguments]
    process = subprocess.Popen(arguments, cwd=PROJECT, env=ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, arguments)

uv_binary = shutil.which('uv')
if uv_binary:
    UV = [uv_binary]
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'], check=True)
    UV = [sys.executable, '-m', 'uv']
command(UV + ['sync', '--locked', '--python', '3.12.11', '--extra', 'dev', '--extra', 'analysis', '--extra', 'model'])
PYTHON = PROJECT / '.venv/bin/python'
JREFLECT = PROJECT / '.venv/bin/jreflect'
UPSTREAM = PROJECT / 'vendor/jpp_lens'
PIN = 'd7efc0de372e20589e899188c0cfe8434acc63ec'
if not UPSTREAM.exists():
    command(['git', 'clone', 'https://github.com/safety-research/jpp_lens.git', UPSTREAM])
dirty = subprocess.check_output(['git', '-C', str(UPSTREAM), 'status', '--porcelain', '--untracked-files=no'], text=True)
if dirty:
    raise RuntimeError('The upstream checkout has local changes; preserve them before using a clean pinned checkout.')
command(['git', '-C', UPSTREAM, 'checkout', PIN])

def run(*arguments):
    command([JREFLECT, *arguments])

def read_json(path):
    return json.loads(Path(path).read_text())


In [ ]:
run('doctor')
probe = """
import json, torch
result = {'cuda_available': torch.cuda.is_available(), 'torch': torch.__version__, 'cuda_runtime': torch.version.cuda}
if result['cuda_available']:
    properties = torch.cuda.get_device_properties(0)
    result.update(gpu=properties.name, vram_gib=round(properties.total_memory / 2**30, 1), bf16=torch.cuda.is_bf16_supported())
    value = torch.ones((16, 16), device='cuda', dtype=torch.bfloat16)
    result['cuda_matmul_sum'] = float((value @ value).sum().cpu())
print(json.dumps(result))
"""
hardware = json.loads(subprocess.check_output([str(PYTHON), '-c', probe], cwd=PROJECT, env=ENV, text=True))
print(json.dumps(hardware, indent=2))
assert hardware['cuda_available'] and hardware['bf16'], 'A working CUDA/bf16 environment is required.'
assert hardware['cuda_matmul_sum'] == 4096.0
if hardware['vram_gib'] < 39:
    print('This is below the planned 40–80 GB GPU range. Review memory requirements before the full lens fit.')


## 4. Local mechanics checks

Expected: 39 passing tests, then a small random-model smoke run. The smoke run trains only a tiny CPU model. Each CLI invocation is a separate process, so models do not remain resident between stages.


In [ ]:
command([PYTHON, '-m', 'pytest', '-q'])
run('smoke', '--output', PROJECT / 'artifacts/notebook-smoke', '--upstream', UPSTREAM)


## 5. Persistence and optional restore

Notebook files and runtime disks are different. Colab virtual machines can be deleted; mounted Drive is also a poor place for thousands of individual checkpoint files. Keep the live run local and copy it as an archive, or use a provider's persistent POSIX disk.

For Colab backups, set `USE_GOOGLE_DRIVE = True`. Otherwise set `BACKUP_DIR` to an already mounted persistent folder. Backups run only at completed stage boundaries, preserve optimizer/Jacobian state, and can be large. A runtime loss between backups loses work since the most recent backup. Use persistent compute for long fits if that loss is unacceptable.

Sources: [Colab runtime and storage FAQ](https://research.google.com/colaboratory/faq.html), [uv/Jupyter environments](https://docs.astral.sh/uv/guides/integration/jupyter/).


In [ ]:
USE_GOOGLE_DRIVE = False  # Set True for optional Colab Drive backups.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = Path('/content/drive/MyDrive/JReflect/backups')
if BACKUP_DIR is not None:
    BACKUP_DIR = Path(BACKUP_DIR).resolve()
    if BACKUP_DIR.is_relative_to(RUN):
        raise ValueError('The backup directory must be outside the live run.')
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
else:
    print('No external backup configured. Artifacts remain on the current runtime disk.')

def backup_run(stage):
    if BACKUP_DIR is None or not (RUN / 'run.json').exists():
        return
    destination = BACKUP_DIR / (RUN.name + '-latest.tar')
    temporary = destination.with_suffix('.tar.partial')
    info = {'run_path': str(RUN), 'project_path': str(PROJECT), 'stage': stage,
            'created_utc': datetime.now(timezone.utc).isoformat()}
    try:
        with tarfile.open(temporary, 'w') as bundle:
            payload = json.dumps(info).encode()
            header = tarfile.TarInfo('backup-info.json')
            header.size = len(payload)
            bundle.addfile(header, io.BytesIO(payload))
            bundle.add(RUN, arcname=str(RUN.relative_to(PROJECT)))
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    print('Saved backup:', destination)


In [ ]:
RESTORE_FROM = None  # Example: BACKUP_DIR / 'notebook-run-latest.tar'
if RESTORE_FROM is not None:
    if RUN.exists() and any(RUN.iterdir()):
        raise RuntimeError('Restore only into a fresh, empty run directory; existing results were not overwritten.')
    with tarfile.open(RESTORE_FROM, 'r') as bundle:
        info = json.load(bundle.extractfile('backup-info.json'))
        if info['run_path'] != str(RUN) or info['project_path'] != str(PROJECT):
            raise RuntimeError('Restore at the original project/run paths so upstream checkpoint paths stay valid.')
        members = [m for m in bundle.getmembers() if m.name != 'backup-info.json']
        extract_regular_files(bundle, PROJECT, prefix=RUN.relative_to(PROJECT).parts, members=members)
    print('Restored stage:', info['stage'])
else:
    print('Starting fresh or continuing the current live run.')


## 6. Prepare the data and try ten problems

Preparation streams only the GSM8K training split and pins model/dataset commits. Defaults are 6,973 train, 300 dev-tuning, and 200 dev-gate examples. On resume, the saved configuration is reused.

The next generation cell downloads the 7B weights on first use. It uses a clearly labeled **logit lens**, not a fitted J++ lens. If it runs out of memory, stop and inspect the error before attempting fitting; do not silently change precision or quantize one arm.


In [ ]:
if not (RUN / 'run.json').exists():
    run('prepare', '--config', PROJECT / 'configs/initial.toml', '--run', RUN)
settings = read_json(RUN / 'run.json')['config']
SEEDS = settings['training']['seeds']
ARMS = ['readout', 'no_readout', 'shuffled']
LENS = RUN / 'lens/lens.pt'
print('Model:', settings['model']['name'])
print('Seeds:', SEEDS, '| Total adapters:', len(SEEDS) * len(ARMS))
backup_run('prepared')


In [ ]:
run('generate', '--run', RUN, '--split', 'dev_tune', '--limit', 10)
backup_run('ten-problem-check')


**Checkpoint:** inspect an example below. Verify that the previous answer is a bare number, readouts contain both `last_prompt` and `first_answer`, and the method is `logit`. A format violation can be a genuine base-model failure; it is recorded rather than corrected by hand.


In [ ]:
row_path = next((RUN / 'base/logit/dev_tune/rows').glob('*.json'))
row = read_json(row_path)['row']
print('Question:', row['question'])
print('First answer:', repr(row['first']['text']))
print('Grade:', row['first_grade'])
print('Readout methods:', list(row['readouts']))
print('Positions:', sorted({r['position'] for r in row['readouts']['logit']}))
print('Concepts:', row['concepts']['logit'])


## 7. Fit the J++ lens, stage by stage

These are the first expensive stages. Measure memory/time on your A100; the ten-problem check does not establish fitting memory. Completed stage files are reused. Jacobians save checkpoints every four prompts; router and expert-weight optimization restart their unfinished stage.

Once the dev gate has been opened, the package closes lens fitting for that experiment. On resume, use the completed markers and continue with the unfinished stage.


In [ ]:
def fit_stage(stage, completed_file):
    if (RUN / 'lens' / completed_file).exists():
        print('Reusing completed stage:', stage)
    else:
        run('jpp-fit', '--run', RUN, '--upstream', UPSTREAM, '--stage', stage)
    backup_run('lens-' + stage)

fit_stage('router', 'router.pt')


In [ ]:
fit_stage('jacobians', 'experts.pt')


In [ ]:
fit_stage('weights', 'lens.meta.json')


In [ ]:
fit_stage('validate', 'validation.json')
validation = read_json(RUN / 'lens/validation.json')
print(json.dumps(validation, indent=2))
if not validation['go']:
    raise RuntimeError('Lens validation did not pass. Keep this diagnostic run; do not proceed to training.')


## 8. Measure GSM8K readout signal

Generate both lenses from the same frozen-base activations. The correctness probe fits on `dev_tune` and evaluates once on `dev_gate`. A saved no-go is a result; it is not bypassed by rerunning the cell.


In [ ]:
run('generate', '--run', RUN, '--split', 'dev_tune', '--lens', LENS)
backup_run('dev-tune-readouts')


In [ ]:
run('generate', '--run', RUN, '--split', 'dev_gate', '--lens', LENS)
backup_run('dev-gate-readouts')


In [ ]:
if not (RUN / 'gate.json').exists():
    run('gate', '--run', RUN)
gate = read_json(RUN / 'gate.json')
print(json.dumps({k: v for k, v in gate.items() if k != 'prediction_rows'}, indent=2))
backup_run('development-gate')
if not gate['go']:
    raise RuntimeError('Development gate is no-go/inconclusive. Preserve the run and stop before full training.')


## 9. Build matching controls and train

All three conditions use the same training examples and targets. Start with ten optimizer steps for the Readout arm; the following loop resumes that adapter and completes every configured arm/seed. Keep the runtime connected through a stage, and retain a persistent backup.


In [ ]:
run('generate', '--run', RUN, '--split', 'train', '--lens', LENS)
if not (RUN / 'examples.json').exists():
    run('examples', '--run', RUN)
backup_run('training-examples')


In [ ]:
pilot_status = RUN / 'adapters' / str(SEEDS[0]) / 'readout/training.json'
if not pilot_status.exists():
    run('train', '--run', RUN, '--arm', 'readout', '--seed', SEEDS[0], '--max-steps', 10)
print(read_json(pilot_status))
backup_run('training-pilot')


In [ ]:
for seed in SEEDS:
    for arm in ARMS:
        status = RUN / 'adapters' / str(seed) / arm / 'training.json'
        if not status.exists() or not read_json(status)['complete']:
            run('train', '--run', RUN, '--arm', arm, '--seed', seed)
        backup_run(f'trained-{seed}-{arm}')


## 10. Review development results

Use these results to assess formatting, change rates, and the planned training settings. Any changed experiment needs a new identity; do not modify a frozen run. These are dev metrics, not final evidence.


In [ ]:
for seed in SEEDS:
    for arm in ARMS:
        run('predict', '--run', RUN, '--split', 'dev_tune', '--arm', arm, '--seed', seed)
run('report', '--run', RUN, '--split', 'dev_tune')
backup_run('development-report')
from IPython.display import display, Markdown, Image
print('Dev metrics:', RUN / 'reports/dev_tune/metrics.json')
display(Markdown((RUN / 'reports/dev_tune/report.md').read_text().split('![Net accuracy changes]')[0]))
display(Image(filename=str(RUN / 'reports/dev_tune/accuracy.png')))


## 11. Freeze and run the final test

Run these cells after reviewing dev. Freeze locks the adapters, lens, configuration, code, environment, and statistical protocol. The final test contains 1,319 items and uses one shared first-attempt cache. Training is closed after freezing.


In [ ]:
if not (RUN / 'protocol.json').exists():
    run('freeze', '--run', RUN)
run('prepare-test', '--run', RUN)
run('generate', '--run', RUN, '--split', 'test', '--lens', LENS)
backup_run('frozen-test-first-attempts')


In [ ]:
for seed in SEEDS:
    for arm in ARMS:
        run('predict', '--run', RUN, '--split', 'test', '--arm', arm, '--seed', seed)
        backup_run(f'test-{seed}-{arm}')
run('report', '--run', RUN, '--split', 'test')
backup_run('final-report')
display(Markdown((RUN / 'reports/test/report.md').read_text().split('![Net accuracy changes]')[0]))
display(Image(filename=str(RUN / 'reports/test/accuracy.png')))


## Results and resuming

Results are under `RUN / 'reports/test'`: Markdown, CSV, full metric JSON, and a PNG. Item-level prediction and readout caches remain in the run directory.

To resume after runtime loss: upload the same source archive, rerun setup with the same Python/dependencies and workspace path, mount persistent storage, set `RESTORE_FROM` to the saved archive, then continue. Do not change code or package versions mid-run. Backups represent completed stages; a lost runtime cannot recover later files that never reached persistent storage.

**Common notebook pitfall:** `!source .venv/bin/activate` does not activate later cells. This notebook always invokes the environment's absolute executable path, so no kernel switching is required.

**Optional extension:** after the primary evaluation, use the CLI's `--ablation readout_removed` or `--ablation tuned_first` for separately labeled exploratory predictions. See `docs/run-guide.md` in the source bundle.
